# 🚀 ApexChess Cloud GPU Training Kernel (NVIDIA T4 / P100)

This automated Kaggle kernel trains the **ApexChess Dual-NNUE neural model** on free cloud GPUs.
- **Hardware**: NVIDIA Tesla T4 (16 GB VRAM) / P100
- **Dataset**: `ssingh22/chess-evaluations` (tactics configuration, 2.6M positions)
- **Architecture**: Dual-NNUE (40,960 -> 256 -> 32 -> 32 -> 1) with SCReLU activations
- **Output**: Exported INT16/INT8 quantized weights and PyTorch checkpoint saved to `/kaggle/working/`

In [ ]:
# 1. Hardware Inspection
import torch
print("=" * 60)
print("  APEX CHESS KAGGLE HARDWARE INSPECTION")
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU Device:      {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("WARNING: No GPU detected! Make sure Accelerator is set to GPU in Kaggle settings.")
!nvidia-smi


In [ ]:
# 2. Clone Repository & Install Dependencies
import os

if not os.path.exists("apex-chess-engine"):
    !git clone https://github.com/hammadshakeelai/apex-chess-engine.git

%cd apex-chess-engine
!git pull origin master
!pip install -q python-chess pyarrow


In [ ]:
# 3. Execute Cloud Training Pipeline (1,000,000 Positions)
# Scaled cloud training run on NVIDIA Tesla GPU
!python scripts/cloud_train_colab.py \
    --positions 1000000 \
    --epochs 15 \
    --batch_size 2048 \
    --save /kaggle/working/apex_cloud_1m.pt \
    --export /kaggle/working/apex_cloud_1m_quant.npz

# Mirror to apex_cloud_v1 for backward compatibility
!cp /kaggle/working/apex_cloud_1m.pt /kaggle/working/apex_cloud_v1.pt
!cp /kaggle/working/apex_cloud_1m_quant.npz /kaggle/working/apex_cloud_v1_quant.npz


In [ ]:
# 4. Verify Exported Weights in Working Directory
import os

print("=" * 60)
print("  KAGGLE WORKING DIRECTORY OUTPUTS")
print("=" * 60)
for f in os.listdir("/kaggle/working"):
    p = os.path.join("/kaggle/working", f)
    size_mb = os.path.getsize(p) / (1024 * 1024)
    print(f"{f:30} : {size_mb:6.2f} MB")
print("=" * 60)
print("Ready for CLI download via: python scripts/kaggle_hub.py --pull")
